# Test sintetico di pycombat

Questo notebook non usa i dati originali. Genera invece un dataset controllato in cui sappiamo gia cosa dovrebbe succedere:

- esiste un effetto clinico vero da preservare (`STABILE` / `INSTABILE`);
- esiste un effetto batch artificiale da rimuovere (`Scanner_A`, `Scanner_B`, `Scanner_C`);
- le feature sono numeriche, come in una matrice radiomica.

Vogliamo assicurarci che:
1. ComBat **riduce** il segnale del batch? 
2. ComBat **preserva** il segnale clinico che gli chiediamo di mantenere?

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
helper_candidates = [cwd, cwd / "notebooks", *[parent / "notebooks" for parent in cwd.parents]]
helper_dir = next(
    candidate for candidate in helper_candidates
    if (candidate / "pycombat_synthetic_helpers.py").exists()
)
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

from pycombat_synthetic_helpers import (
    apply_combat,
    array_quality,
    compute_diagnostics,
    create_synthetic_dataset,
    dataset_preview,
    load_combat_class,
    plot_feature_boxplots,
    plot_pca_before_after,
    plot_pvalue_diagnostics,
)

Combat = load_combat_class()
print(f"Helper caricato da: {helper_dir}")
print(f"Classe Combat: {Combat}")

## 1. Dataset sintetico

1. Creiamo 135 pazienti finti
2. Divido i pazienti in 3 batch/scanner. 

Prime 8 feature: vero effetto clinico. 

Tutte le 40 feature ricevono invece uno shift e una scala diversi per batch (contaminazione che ComBat dovrebbe togliere).

In [ ]:
data = create_synthetic_dataset(random_state=42)

print(f"Campioni: {data.Y.shape[0]}")
print(f"Feature: {data.Y.shape[1]}")
print(f"Feature con segnale clinico simulato: {data.n_signal_features}")
dataset_preview(data)

## 2. Applicazione di ComBat

Passiamo lo scanner in `b`, cioe la variabile di batch da correggere. Passiamo invece `STABILE` / `INSTABILE` in `X`, cioe tra gli effetti di interesse da preservare.

Questa distinzione e importante: se una variabile clinica viene messa in `X`, ComBat cerca di non eliminarla mentre rimuove gli effetti sistematici del batch.

In [ ]:
Y_after, combat = apply_combat(data)

array_quality(Y_after)

## 3. Metriche numeriche

Questa tabella conta quante feature superano una soglia statistica (`p < 0.05`) prima e dopo ComBat.

- `media diversa tra batch`: misura l'effetto additivo dello scanner. Deve scendere.
- `varianza diversa tra batch`: misura l'effetto moltiplicativo/scalare dello scanner. Deve scendere.
- `cliniche simulate significative`: sono le prime 8 feature con segnale clinico vero. Idealmente restano significative.
- `non cliniche falsamente significative`: dovrebbero restare basse.

In [ ]:
diagnostics = compute_diagnostics(data, Y_after)
diagnostics.summary

## 4. PCA prima/dopo

La PCA non sa nulla di batch o clinica: cerca solo le direzioni in cui le feature variano di piu.

Nel dataset sintetico abbiamo aggiunto uno shift forte e coerente per scanner su molte feature. Questo significa che, prima di ComBat, una parte enorme della varianza globale e spiegata dal batch; quindi nella PCA i punti tendono a separarsi per colore/scanner.

Dopo ComBat ci aspettiamo meno separazione per scanner perche abbiamo rimosso proprio quella componente sistematica. Non e una prova matematica definitiva, ma e un buon controllo visivo: se i colori restano separati nettamente, probabilmente il batch effect non e stato rimosso bene. Se rimane una separazione per forma (`STABILE` / `INSTABILE`), non e necessariamente un problema: quel segnale e stato passato in `X` e quindi chiesto a ComBat di preservarlo.

In [ ]:
plot_pca_before_after(data, Y_after)

## 5. Boxplot di una feature

Qui guardiamo una singola feature. Prima di ComBat i batch possono avere mediane e dispersioni diverse; dopo ComBat dovrebbero diventare piu confrontabili. Il boxplot serve a vedere il meccanismo su una feature concreta, mentre la tabella sopra lo riassume su tutte le feature.

In [ ]:
## 6. P-value batch e clinica

Questo grafico controlla separatamente due cose: quanto ogni feature e ancora legata al batch e quanto ogni feature e legata al tipo clinico simulato.

Per ogni feature calcoliamo due test statistici:

- nel pannello sinistro usiamo un test ANOVA tra `Scanner_A`, `Scanner_B` e `Scanner_C`; se il p-value e piccolo, quella feature ha valori diversi tra scanner, quindi contiene batch effect;
- nel pannello destro usiamo un t-test tra `STABILE` e `INSTABILE`; se il p-value e piccolo, quella feature contiene segnale clinico.

Nel grafico non disegniamo direttamente `p`, ma `-log10(p)`. Questo serve solo a rendere leggibili i p-value piccoli: piu il punto e alto, piu l'associazione e forte. La linea tratteggiata e `p = 0.05`; i valori sopra la linea sono significativi secondo questa soglia.

Come leggerlo:

- **Pannello batch**: prima di ComBat, la curva rossa dovrebbe essere alta per molte feature, perche abbiamo creato apposta differenze tra scanner. Dopo ComBat, la curva blu dovrebbe scendere sotto la soglia: significa che le feature non distinguono piu bene gli scanner.
- **Pannello clinico**: le prime 8 feature, evidenziate in grigio, sono quelle in cui abbiamo inserito un vero effetto `STABILE` / `INSTABILE`. Dopo ComBat ci aspettiamo che restino alte o comunque simili a prima, perche quel segnale e stato passato a ComBat in `X` e quindi deve essere preservato.
- Le feature fuori dalla banda grigia non hanno segnale clinico simulato: se diventano alte, sono possibili falsi positivi.

Questo e un controllo diagnostico, non una validazione clinica: qui i p-value non sono corretti per confronti multipli e servono soprattutto a vedere se ComBat sta rimuovendo il batch senza cancellare il segnale che gli abbiamo chiesto di conservare.

## 6. P-value batch e clinica

Questo grafico mostra `-log10(p)`: piu il valore e alto, piu l'associazione e forte. La linea tratteggiata corrisponde a `p = 0.05`.

Nel pannello batch vogliamo vedere la curva dopo ComBat scendere sotto soglia. Nel pannello clinico vogliamo che le prime 8 feature, evidenziate in grigio, restino sopra soglia o comunque non crollino: sono il segnale clinico simulato che abbiamo chiesto di preservare.

In [ ]:
plot_pvalue_diagnostics(data, diagnostics)

## Interpretazione attesa

Su questi dati controllati un risultato sensato e questo:

- le feature associate al batch scendono molto, idealmente vicino a zero;
- le feature cliniche simulate restano rilevabili;
- non compaiono `NaN` o `inf`;
- nella PCA i colori dei batch si mescolano di piu dopo ComBat.

Questo non dimostra che ComBat funzionera automaticamente su qualunque dataset reale, ma dimostra che il modulo si comporta bene quando il problema e noto e controllato.